# Inspecting & verifying circuit construction

This notebook lets you **see and verify** exactly what circuit each
`CircuitConfig` builds. It works three ways at once:

1. **Inline reimplementation**: we rebuild the trace logic from scratch in
   plain cells (setup gates, layer gates, measurements), so the mechanics are
   visible, not hidden behind a function call.
2. **Cross-check against the library**: we run the real
   [`trace_circuit`](../../src/sparsegf2/circuits/inspector.py) /
   `inspect_circuit` and assert our inline version produces the *same*
   structured operations. If they ever disagree, a cell fails loudly.
3. **Comprehensive sweep**: we inspect **every** construction (all pictures
   × graphs × gating × matching × measurement modes) and run structural
   checks (gate counts, valid edges, reference qubits never measured, gated
   measurements only on touched qubits, …).

The guiding distinction throughout: **deterministic** construction gates (the
Bell pairs that build purification / single_ref) are shown as the *actual*
gates (`H`, `CX`); **random** two-qubit Cliffords are shown as `C[index]`
(the Sp(4) table index, so you can verify reproducibility); and the **random
selection** of measured qubits is flagged as such.

## Setup

In [1]:
import numpy as np
from sparsegf2 import code_dimension, entanglement_entropy, SparseGF2
from sparsegf2.circuits import (
    CircuitConfig, CircuitBuilder, trace_circuit, inspect_circuit, setup_ops,
)
from sparsegf2.circuits.picture import setup_picture
from sparsegf2.circuits.ops import apply_named_gate
print('ready')


ready


## Part A: the deterministic setup gates (picture construction)

A *picture* prepares the initial state with a fixed sequence of gates. These
are **deterministic** (the same every run), so the inspector shows the actual
gates. Let's derive them from first principles, inline:

- `pure_state`: nothing; the simulator starts in $|0\dots0\rangle$.
- `purification`: for each system qubit $i$, a Bell pair with reference
  $i+n$: `H(i); CX(i, i+n)`.
- `single_ref`: one Bell pair between the last system qubit $n-1$ and the
  reference $n$: `H(n-1); CX(n-1, n)`.

In [2]:
def my_setup_ops(picture, n):
    '''Inline reconstruction of each picture's deterministic setup gates.'''
    picture = str(picture)
    if picture == 'pure_state':
        return []
    if picture == 'purification':
        ops = []
        for i in range(n):
            ops.append(('H', (i,)))            # put system qubit i in |+>
            ops.append(('CX', (i, i + n)))     # entangle it with reference i+n
        return ops
    if picture == 'single_ref':
        return [('H', (n - 1,)), ('CX', (n - 1, n))]   # one Bell pair: (n-1, n)
    raise ValueError(picture)

# Cross-check: our inline recipe must match the library's setup_ops exactly.
for pic in ('pure_state', 'purification', 'single_ref'):
    lib = [(o.label, o.qubits) for o in setup_ops(pic, 4)]
    mine = my_setup_ops(pic, 4)
    assert lib == mine, (pic, lib, mine)
    print(f'{pic:>13}: {mine}')
print('\n✓ inline my_setup_ops matches library setup_ops for all pictures')


   pure_state: []
 purification: [('H', (0,)), ('CX', (0, 4)), ('H', (1,)), ('CX', (1, 5)), ('H', (2,)), ('CX', (2, 6)), ('H', (3,)), ('CX', (3, 7))]
   single_ref: [('H', (3,)), ('CX', (3, 4))]

✓ inline my_setup_ops matches library setup_ops for all pictures


### A.2: the setup gates really build the right state

Showing the gates is one thing; we also verify they *do the right thing*.
Apply our inline setup ops by hand and check the resulting state:

- **purification**: every system qubit entangled with its reference ⇒ code
  dimension $k = S(\text{system}) = n$.
- **single_ref**: the reference is entangled with the system ⇒
  $S(\text{reference}) = 1$, and it is paired specifically with qubit $n-1$
  (measuring $n-1$ should purify it; measuring any other system qubit should
  not).

In [3]:
n = 6
# Build purification by hand from the inline recipe, then check k = n.
sim = SparseGF2(2 * n)
for label, qs in my_setup_ops('purification', n):
    apply_named_gate(sim, label, qs)
print('purification: code_dimension k =', code_dimension(sim, n), '(expect', n, ')')
assert code_dimension(sim, n) == n

# single_ref: reference paired with qubit n-1 specifically.
sim = SparseGF2(n + 1)
for label, qs in my_setup_ops('single_ref', n):
    apply_named_gate(sim, label, qs)
print('single_ref : S(reference) =', entanglement_entropy(sim, [n]), '(expect 1)')
assert entanglement_entropy(sim, [n]) == 1
s_after_paired = sim.copy(); s_after_paired.measure_z(n - 1)
s_after_other = sim.copy(); s_after_other.measure_z(0)
print('  measuring the PAIRED qubit n-1 -> S(ref) =', entanglement_entropy(s_after_paired, [n]), '(purifies)')
print('  measuring an OTHER qubit 0     -> S(ref) =', entanglement_entropy(s_after_other, [n]), '(unchanged)')
assert entanglement_entropy(s_after_paired, [n]) == 0
assert entanglement_entropy(s_after_other, [n]) == 1
print('\n✓ Bell pairs are wired to exactly the right qubits')


purification: code_dimension k = 6 (expect 6 )
single_ref : S(reference) = 1 (expect 1)
  measuring the PAIRED qubit n-1 -> S(ref) = 0 (purifies)
  measuring an OTHER qubit 0     -> S(ref) = 1 (unchanged)

✓ Bell pairs are wired to exactly the right qubits


And here is how the library renders that setup (purification, n=4):

In [4]:
cfg = CircuitConfig(graph_spec='cycle', n=4, picture='purification', p=0.1, depth_factor=1)
tr = trace_circuit(cfg, max_layers=0)
setup_stage = tr.stages[0]
from sparsegf2.circuits.inspector import _render_stage
print('\n'.join(_render_stage(setup_stage)))


setup (deterministic)
  H     q0
  CX    q0→q4   # Bell pair: system 0 ↔ reference 4
  H     q1
  CX    q1→q5   # Bell pair: system 1 ↔ reference 5
  H     q2
  CX    q2→q6   # Bell pair: system 2 ↔ reference 6
  H     q3
  CX    q3→q7   # Bell pair: system 3 ↔ reference 7


## Part B: the layer schedule (random Cliffords + measurements)

Each circuit layer comes from the scheduler as a `CircuitLayer` with three
fields: `gate_pairs` (which qubit pairs get a gate), `cliff_indices` (the
**random** Sp(4) table index for each gate), and `meas_qubits` (which qubits
are measured). Let's turn a layer into operations inline, then confirm it
matches the library trace.

In [5]:
def my_layer_ops(layer):
    '''Inline reconstruction of a layer's operations.'''
    ops = []
    for g, (qi, qj) in enumerate(layer.gate_pairs):
        ci = int(layer.cliff_indices[g])
        ops.append(('gate', f'C[{ci}]', (int(qi), int(qj)), True))   # random Sp(4) #ci
    fired = set(layer.meas_qubits)
    candidates = layer.meas_candidates or layer.meas_qubits
    for q in candidates:
        label = 'MZ' if int(q) in fired else 'MZ?'   # fired vs candidate
        ops.append(('measure', label, (int(q),), True))
    return ops

cfg = CircuitConfig(graph_spec='cycle', n=8, p=0.3, depth_factor=2)
# Inline: take layer 0 straight from a CircuitBuilder.
layer0 = next(CircuitBuilder(cfg, sample_seed=0).layers())
mine = my_layer_ops(layer0)
# Library: pull layer 0's ops from trace_circuit.
tr = trace_circuit(cfg, sample_seed=0, max_layers=1)
lib_layer = next(s for s in tr.stages if s.kind == 'layer')
lib = [(o.kind, o.label, o.qubits, o.random) for o in lib_layer.ops]
for op in mine:
    print(op)
assert mine == lib
print('\n✓ inline my_layer_ops matches library trace ops exactly')


('gate', 'C[64]', (0, 1), True)
('gate', 'C[557]', (2, 3), True)
('gate', 'C[471]', (4, 5), True)
('gate', 'C[315]', (6, 7), True)
('measure', 'MZ?', (0,), True)
('measure', 'MZ?', (1,), True)
('measure', 'MZ', (2,), True)
('measure', 'MZ?', (3,), True)
('measure', 'MZ?', (4,), True)
('measure', 'MZ?', (5,), True)
('measure', 'MZ', (6,), True)
('measure', 'MZ?', (7,), True)

✓ inline my_layer_ops matches library trace ops exactly


### B.2: random, but reproducible

The Cliffords are *random* (a uniform draw from the 720-element Sp(4) table),
but seeded, so the **same** `(config, sample_seed)` gives the **same**
indices every time, and a different seed gives different ones. The indices in
the trace let you verify this directly.

In [6]:
def cliff_indices(cfg, seed, k=4):
    tr = trace_circuit(cfg, sample_seed=seed, max_layers=k)
    return [o.detail['clifford_index'] for s in tr.stages for o in s.ops if o.kind == 'gate']

cfg = CircuitConfig(graph_spec='cycle', n=8, p=0.2, depth_factor=2)
print('seed 0 :', cliff_indices(cfg, 0))
print('seed 0 :', cliff_indices(cfg, 0), '  (identical, reproducible)')
print('seed 1 :', cliff_indices(cfg, 1), '  (different, independent realization)')
assert cliff_indices(cfg, 0) == cliff_indices(cfg, 0)
assert cliff_indices(cfg, 0) != cliff_indices(cfg, 1)
print('\n✓ same seed reproduces, different seed varies')

seed 0 : [64, 557, 471, 315, 360, 266, 131, 667, 118, 545, 504, 255, 664, 536, 263, 696]
seed 0 : [64, 557, 471, 315, 360, 266, 131, 667, 118, 545, 504, 255, 664, 536, 263, 696]   (identical, reproducible)
seed 1 : [363, 469, 288, 31, 105, 687, 280, 642, 338, 349, 593, 680, 608, 216, 507, 396]   (different, independent realization)

✓ same seed reproduces, different seed varies


## Part C: a from-scratch renderer (and the library's view)

Putting A and B together: a small inline renderer that prints the setup gates
and the first few layers. Then we show the library's `inspect_circuit` output
for the same config, and assert the underlying operations are identical.

In [7]:
def my_render(config, seed=0, max_layers=4):
    '''A minimal from-scratch inspector: setup + first N layers.'''
    lines = [f'picture={config.picture}  graph={config._graph.name}  '
             f'gating={config.gating_mode}  p={config.p}']
    sops = my_setup_ops(str(config.picture), config.n)
    lines.append('  setup: ' + (', '.join(f'{l}{q}' for l, q in sops) or '(none)'))
    for i, layer in zip(range(max_layers), CircuitBuilder(config, seed).layers()):
        g = '  '.join(f'C[{int(layer.cliff_indices[j])}]{tuple(layer.gate_pairs[j])}'
                      for j in range(len(layer.gate_pairs)))
        m = ' '.join(f'q{q}' for q in layer.meas_qubits) or '-'
        lines.append(f'  L{i}: gates[ {g} ]  measZ[ {m} ]')
    return '\n'.join(lines)

cfg = CircuitConfig(graph_spec='cycle', n=8, picture='single_ref', p=0.16, depth_factor=4)
print('################  my_render (inline)  ################')
print(my_render(cfg, seed=0, max_layers=4))
print('\n################  library inspect_circuit  ################')
print(inspect_circuit(cfg, sample_seed=0, max_layers=4))

################  my_render (inline)  ################
picture=single_ref  graph=cycle(8)  gating=brickwork  p=0.16
  setup: H(7,), CX(7, 8)
  L0: gates[ C[64](0, 1)  C[557](2, 3)  C[471](4, 5)  C[315](6, 7) ]  measZ[ q2 q6 ]
  L1: gates[ C[360](0, 7)  C[266](1, 2)  C[131](3, 4)  C[667](5, 6) ]  measZ[ q5 ]
  L2: gates[ C[118](0, 1)  C[545](2, 3)  C[504](4, 5)  C[255](6, 7) ]  measZ[ q5 q6 ]
  L3: gates[ C[664](0, 7)  C[536](1, 2)  C[263](3, 4)  C[696](5, 6) ]  measZ[ q4 ]

################  library inspect_circuit  ################
Circuit inspection: first 4 of 32 measured layers
  picture        : single_ref  (9 qubits: system 0-7, reference [8])
  graph          : cycle(8)
  gating         : brickwork / round_robin
  measurement    : bernoulli  p=0.16
  depth          : O(n) ×4 = 32 measured layers
  exp. gate:meas : 3.12 : 1
  sample_seed    : 0

setup (deterministic)
  H     q7
  CX    q7→q8   # Bell pair: system 7 ↔ reference 8

layer 0
  2q gates [random Sp(4)]:  C[64] (q0,q1) 

Confirm both views describe the *same* circuit (compare structured ops):

In [8]:
def structured(config, seed, k):
    out = []
    out += [('setup',) + t for t in my_setup_ops(str(config.picture), config.n)]
    for i, layer in zip(range(k), CircuitBuilder(config, seed).layers()):
        out += [('L%d' % i,) + (op[1], op[2]) for op in my_layer_ops(layer)]
    return out

def lib_structured(config, seed, k):
    tr = trace_circuit(config, sample_seed=seed, max_layers=k)
    out = []
    for s in tr.stages:
        if s.kind == 'setup':
            out += [('setup', o.label, o.qubits) for o in s.ops]
        elif s.kind == 'layer':
            out += [('L%d' % s.index, o.label, o.qubits) for o in s.ops]
    return out

assert structured(cfg, 0, 4) == lib_structured(cfg, 0, 4)
print('✓ inline and library agree on the full operation list')


✓ inline and library agree on the full operation list


## Part D: comprehensive sweep over EVERY construction

Now the thorough part. We inspect every combination of

- **graph**: cycle, complete
- **picture**: pure_state, purification, single_ref
- **gating**: brickwork (× round_robin / palette / fresh) and random_edge
  (× 1 edge / n/2 edges)
- **measurement**: bernoulli, gated, random_pair

and run structural correctness checks on each:

1. setup ops match our inline recipe;
2. every layer has the expected number of gates ($n/2$ brickwork; $m$ for
   random_edge, capped at the edge count);
3. every gate sits on a real graph edge with distinct qubits;
4. **no reference qubit is ever measured** (only system qubits $0..n-1$);
5. `gated` measurements only touch qubits the layer's gates touched;
6. `random_pair` measures at most 2 qubits.

In [9]:
def check_construction(config, max_layers=6, seed=0):
    '''Return a list of problems (empty list == all checks pass).'''
    n = config.n
    edges = {tuple(e) for e in config._graph.edges}
    tr = trace_circuit(config, sample_seed=seed, max_layers=max_layers)
    issues = []
    setup = next(s for s in tr.stages if s.kind == 'setup')
    if [(o.label, o.qubits) for o in setup.ops] != my_setup_ops(str(config.picture), n):
        issues.append('setup mismatch')
    if config.gating_mode == 'brickwork':
        exp_g = n // 2
    else:
        exp_g = min(config.resolved_gates_per_layer(), len(edges))
    for s in tr.stages:
        if s.kind != 'layer':
            continue
        gates = [o for o in s.ops if o.kind == 'gate']
        meas = [o for o in s.ops if o.kind == 'measure']
        if len(gates) != exp_g:
            issues.append(f'{s.label}: {len(gates)} gates != {exp_g}')
        for o in gates:
            u, v = o.qubits
            if u == v or (min(u, v), max(u, v)) not in edges:
                issues.append(f'{s.label}: bad edge {o.qubits}')
        for o in meas:
            if o.qubits[0] >= n:
                issues.append(f'{s.label}: measured REFERENCE qubit {o.qubits[0]}')
        if config.measurement_mode == 'gated':
            touched = {q for o in gates for q in o.qubits}
            if any(o.qubits[0] not in touched for o in meas):
                issues.append(f'{s.label}: gated measured an untouched qubit')
        if config.measurement_mode == 'random_pair' and len(meas) > 2:
            issues.append(f'{s.label}: random_pair measured >2')
    return issues
print('check_construction defined')


check_construction defined


In [10]:
pictures = ['pure_state', 'purification', 'single_ref']
gatings = [('brickwork', {}), ('random_edge', {'gates_per_layer': 1}),
           ('random_edge', {'gates_per_layer': 4})]
measurements = ['bernoulli', 'gated', 'random_pair']

results = {}
for graph in ('cycle', 'complete'):
    for pic in pictures:
        for gate, extra in gatings:
            matchings = ['round_robin', 'palette', 'fresh'] if gate == 'brickwork' else ['-']
            for mm in matchings:
                for meas in measurements:
                    kw = dict(graph_spec=graph, n=8, picture=pic, gating_mode=gate,
                              measurement_mode=meas, p=0.3, depth_factor=1, **extra)
                    if gate == 'brickwork':
                        kw['matching_mode'] = mm
                    cfg = CircuitConfig(**kw)
                    results[(graph, pic, gate, str(extra.get('gates_per_layer', '')), mm, meas)] = \
                        check_construction(cfg)

fails = {k: v for k, v in results.items() if v}
print(f'Checked {len(results)} constructions (2 graphs x 3 pictures x gating x matching x measurement).')
print(f'PASS: {len(results) - len(fails)} / {len(results)}')
if fails:
    for k, v in fails.items():
        print('  FAIL', k, '->', v)
else:
    print('\n✓ EVERY construction is structurally correct')
assert not fails

Checked 90 constructions (2 graphs x 3 pictures x gating x matching x measurement).
PASS: 90 / 90

✓ EVERY construction is structurally correct


Here is the full pass matrix for the **cycle** graph (the nearest-neighbor model), for eyeballing:

In [11]:
print(f"{'picture':>13} {'gating':>22} {'matching':>11} {'measure':>11}  result")
print('-' * 72)
for (graph, pic, gate, m, mm, meas), issues in results.items():
    if graph != 'cycle':
        continue
    g = gate + (f'(m={m})' if gate == 'random_edge' else '')
    status = 'PASS' if not issues else 'FAIL'
    print(f'{pic:>13} {g:>22} {mm:>11} {meas:>11}  {status}')


      picture                 gating    matching     measure  result
------------------------------------------------------------------------
   pure_state              brickwork round_robin   bernoulli  PASS
   pure_state              brickwork round_robin       gated  PASS
   pure_state              brickwork round_robin random_pair  PASS
   pure_state              brickwork     palette   bernoulli  PASS
   pure_state              brickwork     palette       gated  PASS
   pure_state              brickwork     palette random_pair  PASS
   pure_state              brickwork       fresh   bernoulli  PASS
   pure_state              brickwork       fresh       gated  PASS
   pure_state              brickwork       fresh random_pair  PASS
   pure_state       random_edge(m=1)           -   bernoulli  PASS
   pure_state       random_edge(m=1)           -       gated  PASS
   pure_state       random_edge(m=1)           - random_pair  PASS
   pure_state       random_edge(m=4)           -   ber

## Part D.2: full renderings of representative constructions

The checks above are automated; here are full inspector renderings so you can
*read* a representative circuit from each major family and confirm by eye.

In [12]:
examples = [
    ('pure_state, cycle, brickwork/round_robin, bernoulli',
     dict(graph_spec='cycle', n=8, picture='pure_state', p=0.16, depth_factor=2)),
    ('purification, cycle, brickwork/round_robin, bernoulli',
     dict(graph_spec='cycle', n=6, picture='purification', p=0.16, depth_factor=2)),
    ('single_ref, cycle, brickwork/round_robin, bernoulli',
     dict(graph_spec='cycle', n=8, picture='single_ref', p=0.16, depth_factor=2)),
    ('pure_state, cycle, random_edge (1 edge), bernoulli',
     dict(graph_spec='cycle', n=8, gating_mode='random_edge', gates_per_layer=1, p=0.1, depth_factor=2)),
    ('pure_state, cycle, random_edge (n/2 edges), bernoulli',
     dict(graph_spec='cycle', n=8, gating_mode='random_edge', gates_per_layer=4, p=0.1, depth_factor=2)),
    ('pure_state, complete, brickwork/fresh, gated',
     dict(graph_spec='complete', n=6, matching_mode='fresh', measurement_mode='gated', p=0.3, depth_factor=2)),
    ('pure_state, cycle, brickwork, random_pair',
     dict(graph_spec='cycle', n=8, measurement_mode='random_pair', p=0.5, depth_factor=2)),
]
for title, kw in examples:
    print('#' * 78)
    print('##', title)
    print('#' * 78)
    print(inspect_circuit(CircuitConfig(**kw), sample_seed=0, max_layers=3))
    print()


##############################################################################
## pure_state, cycle, brickwork/round_robin, bernoulli
##############################################################################
Circuit inspection: first 3 of 16 measured layers
  picture        : pure_state  (8 qubits (all system))
  graph          : cycle(8)
  gating         : brickwork / round_robin
  measurement    : bernoulli  p=0.16
  depth          : O(n) ×2 = 16 measured layers
  exp. gate:meas : 3.12 : 1
  sample_seed    : 0

setup (none: |0…0⟩)

layer 0
  2q gates [random Sp(4)]:  C[64] (q0,q1)   C[557] (q2,q3)   C[471] (q4,q5)   C[315] (q6,q7)
  measure Z  fired [q2 q6]   candidates [q0 q1 q2 q3 q4 q5 q6 q7]

layer 1
  2q gates [random Sp(4)]:  C[360] (q0,q7)   C[266] (q1,q2)   C[131] (q3,q4)   C[667] (q5,q6)
  measure Z  fired [q5]   candidates [q0 q1 q2 q3 q4 q5 q6 q7]

layer 2
  2q gates [random Sp(4)]:  C[118] (q0,q1)   C[545] (q2,q3)   C[504] (q4,q5)   C[255] (q6,q7)
  measure Z  fired 

## Part E: targeted correctness guarantees

A few specific invariants worth calling out explicitly, each verified over
many layers / seeds.

In [13]:
# (1) Reference qubits are NEVER measured (purification + single_ref).
for pic, total in [('purification', 16), ('single_ref', 9)]:
    cfg = CircuitConfig(graph_spec='cycle', n=8, picture=pic, p=1.0, depth_factor=2)
    tr = trace_circuit(cfg, max_layers=16)
    measured = {o.qubits[0] for s in tr.stages for o in s.ops if o.kind == 'measure'}
    assert all(q < 8 for q in measured), (pic, measured)
    print(f'{pic:>13}: measured qubits {sorted(measured)} -- all < 8 (system only) ✓')

# (2) Single-edge random_edge runs O(n^2) layers; n/2 edges matches brickwork.
cb = CircuitConfig(graph_spec='cycle', n=8, depth_factor=2)
c1 = CircuitConfig(graph_spec='cycle', n=8, gating_mode='random_edge', gates_per_layer=1, depth_factor=2)
cn = CircuitConfig(graph_spec='cycle', n=8, gating_mode='random_edge', gates_per_layer=4, depth_factor=2)
print(f'\nbrickwork layers={cb.total_layers()}, single-edge layers={c1.total_layers()} (O(n^2)), '
      f'n/2-edge layers={cn.total_layers()}')
assert c1.total_layers() == 64 and cn.total_layers() == cb.total_layers() == 16
print('✓ depth normalization correct')

# (3) gated only measures touched qubits; random_pair measures <= 2.
cg = CircuitConfig(graph_spec='cycle', n=8, gating_mode='random_edge', gates_per_layer=3,
                   measurement_mode='gated', p=1.0, depth_factor=1)
for s in trace_circuit(cg, max_layers=8).stages:
    if s.kind == 'layer':
        touched = {q for o in s.ops if o.kind == 'gate' for q in o.qubits}
        meas = {o.qubits[0] for o in s.ops if o.kind == 'measure'}
        assert meas <= touched
print('✓ gated measurements are a subset of gate-touched qubits')
cp = CircuitConfig(graph_spec='cycle', n=8, measurement_mode='random_pair', p=1.0, depth_factor=2)
for s in trace_circuit(cp, max_layers=16).stages:
    if s.kind == 'layer':
        assert sum(o.kind == 'measure' for o in s.ops) <= 2
print('✓ random_pair measures at most 2 qubits per layer')


 purification: measured qubits [0, 1, 2, 3, 4, 5, 6, 7] -- all < 8 (system only) ✓
   single_ref: measured qubits [0, 1, 2, 3, 4, 5, 6, 7] -- all < 8 (system only) ✓

brickwork layers=16, single-edge layers=64 (O(n^2)), n/2-edge layers=16
✓ depth normalization correct
✓ gated measurements are a subset of gate-touched qubits
✓ random_pair measures at most 2 qubits per layer


## Summary

- The inline reconstructions (`my_setup_ops`, `my_layer_ops`, `my_render`)
  match the library `setup_ops` / `trace_circuit` / `inspect_circuit` exactly,
  so what the inspector shows is what is actually built.
- Setup gates are the real deterministic construction (Bell pairs wired to
  the right qubits, verified by the resulting entropies); layer gates are
  random Sp(4) (reproducible by index); measured qubits are a random,
  system-only selection.
- **Every** construction across graphs × pictures × gating × matching ×
  measurement passes its structural checks, and the targeted invariants
  (reference never measured, depth normalization, gated/random_pair
  semantics) hold.

To inspect any new config yourself:

```python
from sparsegf2.circuits import inspect_circuit, CircuitConfig
print(inspect_circuit(CircuitConfig(graph_spec='cycle', n=8, picture='single_ref', p=0.16), max_layers=15))
```
or from the shell: `python scripts/inspect_circuit.py --n 8 --picture single_ref --layers 15`.